<a href="https://colab.research.google.com/github/Samarth11-byte/Agents/blob/main/Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
%pip install -q crewai requests litellm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.8/36.8 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.3/278.3 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.3/90.3 kB 6.3 MB/s eta 0:00:00


In [6]:
import requests
from google.colab import userdata

NEXUS_API_KEY = userdata.get("NEXUS_API_KEY")

response = requests.post(
    "https://nexusapi.navigatelabs.ai/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {NEXUS_API_KEY}",
        "Content-Type": "application/json"
    },
    json={
        "model": "nano-banana",
        "messages": [
            {
                "role": "user",
                "content": "Reply with: Nexus is working!"
            }
        ]
    },
    timeout=30
)

print("Status:", response.status_code)

if response.ok:
    print(response.json()["choices"][0]["message"]["content"])
else:
    print(response.text[:500])

Status: 200
Nexus is working!


In [7]:
import requests

response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={
        "latitude": 12.9716,
        "longitude": 77.5946,
        "current": "temperature_2m,precipitation,wind_speed_10m"
    },
    timeout=30
)

print("Status:", response.status_code)

if response.ok:
    print(response.json()["current"])
else:
    print(response.text[:500])


Status: 200
{'time': '2026-10-21T06:30', 'interval': 900, 'temperature_2m': 27.4, 'precipitation': 0.1, 'wind_speed_10m': 17.3}


In [8]:
from crewai.tools import tool
import requests

# TOOL 1: Weather API
@tool("Weather API Tool")
def get_weather(city: str) -> str:
    """Get current weather for Bengaluru using a weather API."""
    response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": 12.9716,
            "longitude": 77.5946,
            "current": "temperature_2m,precipitation,wind_speed_10m"
        },
        timeout=20
    )
    response.raise_for_status()
    weather = response.json()["current"]

    return (
        f"Weather for {city}: "
        f"Temperature {weather['temperature_2m']}°C, "
        f"Precipitation {weather['precipitation']} mm, "
        f"Wind speed {weather['wind_speed_10m']} km/h"
    )


# TOOL 2: Event Schedule Generator
@tool("Event Schedule Tool")
def create_schedule(event_name: str, duration_hours: int) -> str:
    """Create a simple schedule for a college event."""
    if duration_hours < 1:
        return "Error: Event duration must be at least 1 hour."

    return (
        f"Event: {event_name}\n"
        f"Hour 1: Welcome and registration\n"
        f"Hour 2: Main event activities\n"
        f"Hour 3: Interactive session and closing"
        if duration_hours >= 3
        else f"Event: {event_name}\n"
             f"Schedule: Welcome, main activities and closing "
             f"within {duration_hours} hour(s)."
    )


# TOOL 3: Resource Checklist
@tool("Resource Checklist Tool")
def get_resources(attendees: int) -> str:
    """Calculate basic resources required for college attendees."""
    if attendees < 1:
        return "Error: Attendee count must be positive."

    return (
        f"For {attendees} attendees:\n"
        f"Chairs: {attendees}\n"
        f"Water bottles: {attendees}\n"
        f"Registration badges: {attendees}\n"
        f"Suggested volunteers: {(attendees + 19) // 20}"
    )


# TOOL 4: Seating Calculator
@tool("Capacity Calculator Tool")
def calculate_seating(attendees: int) -> str:
    """Calculate seating and suggested extra capacity."""
    if attendees < 1:
        return "Error: Attendee count must be positive."

    extra_seats = (attendees + 9) // 10
    return (
        f"Attendees: {attendees}\n"
        f"Required seats including 10% extra: {attendees + extra_seats}"
    )


# TOOL 5: Announcement Generator
@tool("Announcement Tool")
def create_announcement(event_name: str, date: str, venue: str) -> str:
    """Generate an announcement using supplied event details."""
    return (
        f"📢 COLLEGE EVENT ANNOUNCEMENT\n"
        f"Event: {event_name}\n"
        f"Date: {date}\n"
        f"Venue: {venue}\n"
        f"All students are welcome to participate!"
    )


# TOOL 6: Coordination Checklist
@tool("Coordination Checklist Tool")
def coordination_checklist(attendees: int) -> str:
    """Generate a checklist for coordinating a campus event."""
    if attendees < 1:
        return "Error: Attendee count must be positive."

    return (
        "Coordination Checklist:\n"
        "1. Confirm venue booking.\n"
        "2. Arrange seating and registration.\n"
        "3. Check weather conditions.\n"
        "4. Test microphones and projectors.\n"
        "5. Assign volunteers.\n"
        "6. Keep a first-aid kit available."
    )


print("All 6 tools created successfully!")

for campus_tool in [
    get_weather,
    create_schedule,
    get_resources,
    calculate_seating,
    create_announcement,
    coordination_checklist
]:
    print("-", campus_tool.name)

All 6 tools created successfully!
- Weather API Tool
- Event Schedule Tool
- Resource Checklist Tool
- Capacity Calculator Tool
- Announcement Tool
- Coordination Checklist Tool


In [10]:
import crewai
print("CrewAI version:", crewai.__version__)

CrewAI version: 1.15.28


In [11]:
from crewai import Agent

# AGENT 1: Event Planner
event_planner = Agent(
    role="Campus Event Planner",
    goal="Create schedules and identify resources for college events",
    backstory="An experienced college event planner who organizes campus activities.",
    tools=[create_schedule, get_resources],
    verbose=True
)

# AGENT 2: Campus Coordinator
campus_coordinator = Agent(
    role="Campus Coordinator",
    goal="Manage weather, seating and event arrangements",
    backstory="A campus facilities coordinator responsible for safe and smooth events.",
    tools=[get_weather, calculate_seating, coordination_checklist],
    verbose=True
)

# AGENT 3: Communication Manager
communication_manager = Agent(
    role="Communication Manager",
    goal="Prepare clear announcements for students and faculty",
    backstory="A college communications specialist who shares accurate event information.",
    tools=[create_announcement],
    verbose=True
)

print("All 3 agents created successfully!")

All 3 agents created successfully!


In [12]:
from crewai import Task

# TASK 1: Plan the event
plan_task = Task(
    description=(
        "Plan a college event named {event_name}. "
        "The event lasts {duration} hours and has {attendees} attendees. "
        "Use the Event Schedule Tool and Resource Checklist Tool. "
        "Provide the schedule and required resources."
    ),
    expected_output="Event schedule and resource checklist.",
    agent=event_planner
)

# TASK 2: Coordinate the event
arrangement_task = Task(
    description=(
        "Use the previous event plan. "
        "Check the current weather in Bengaluru using the Weather API Tool. "
        "Calculate seating using the Capacity Calculator Tool. "
        "Use the Coordination Checklist Tool to prepare arrangements. "
        "Summarize the weather, seating and coordination requirements."
    ),
    expected_output="Weather report, seating calculation and coordination checklist.",
    agent=campus_coordinator,
    context=[plan_task]
)

# TASK 3: Prepare the announcement
announcement_task = Task(
    description=(
        "Use the event plan and arrangements from the previous tasks. "
        "Create an announcement for {event_name}, scheduled on {date} "
        "at {venue}. Use the Announcement Tool. "
        "Do not invent missing event details. Include relevant arrangements."
    ),
    expected_output="A complete student announcement with event details and arrangements.",
    agent=communication_manager,
    context=[plan_task, arrangement_task]
)

print("All 3 chained tasks created successfully!")

All 3 chained tasks created successfully!


In [13]:
from crewai import Crew, Process

campus_crew = Crew(
    agents=[
        event_planner,
        campus_coordinator,
        communication_manager
    ],
    tasks=[
        plan_task,
        arrangement_task,
        announcement_task
    ],
    process=Process.sequential,
    verbose=True
)

print("Smart College Campus Assistant crew created!")
print("Agents:", len(campus_crew.agents))
print("Tasks:", len(campus_crew.tasks))

Smart College Campus Assistant crew created!
Agents: 3
Tasks: 3


In [14]:
from google.colab import userdata
from crewai import LLM

api_key = userdata.get("NEXUS_API_KEY")

llm = LLM(
    model="openai/nano-banana",
    base_url="https://nexusapi.navigatelabs.ai/v1",
    api_key=api_key
)

try:
    result = llm.call(
        messages=[
            {
                "role": "user",
                "content": "Reply with exactly: CrewAI Nexus connection works."
            }
        ]
    )
    print("SUCCESS!")
    print(result)

except Exception as e:
    print("ERROR:", str(e)[:800])

SUCCESS!
CrewAI Nexus connection works.


In [15]:
print("LLM model:", llm.model)
print("Function calling supported:", llm.supports_function_calling())

LLM model: nano-banana
Function calling supported: True


In [16]:
import inspect
from crewai import Agent

print(inspect.signature(Agent))

(*, entity_type: Literal['agent'] = 'agent', id: Annotated[uuid.UUID, UuidVersion(uuid_version=4)] = <factory>, role: str, goal: str, backstory: str, config: dict[str, typing.Any] | None = None, cache: bool = True, verbose: bool = False, max_rpm: int | None = None, allow_delegation: bool = False, tools: list[crewai.tools.base_tool.BaseTool] | None = <factory>, max_iter: int = 25, tool_failure_policy: crewai.tools.tool_failure.ToolFailurePolicy | None = None, agent_executor: crewai.agents.crew_agent_executor.CrewAgentExecutor | crewai.experimental.agent_executor.AgentExecutor | None = None, i18n: crewai.utilities.i18n.I18N = <factory>, llm: Annotated[str | crewai.llms.base_llm.BaseLLM | None, BeforeValidator(func=<function _validate_llm_ref at 0x7b72d5634360>, json_schema_input_type=PydanticUndefined), PlainSerializer(func=<function _serialize_llm_ref at 0x7b72d5634680>, return_type=dict | None, when_used='json')] = None, crew: Annotated[crewai.crew.Crew | str | None, BeforeValidator(fu

In [18]:
import asyncio

try:
    test_result = await test_agent.aexecute_task(test_task)

    print("\nTEST RESULT:")
    print(test_result)

except Exception as e:
    print("\nTEST FAILED:")
    print(type(e).__name__, str(e)[:1000])

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Campus Assistant                                                                                        │
│                                                                                                                 │
│  Task: Reply with exactly: The campus assistant is ready.                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Campus Assistant                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The campus assistant is ready.                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


TEST RESULT:
The campus assistant is ready.


In [21]:
# Run all six tools directly through Python

event_name = "AI Innovation Day"
duration = 3
attendees = 50
date = "25 October 2026"
venue = "College Auditorium"

print("===== TOOL 1: EVENT SCHEDULE =====")
print(create_schedule.run(
    event_name=event_name,
    duration_hours=duration
))

print("\n===== TOOL 2: RESOURCE CHECKLIST =====")
print(get_resources.run(attendees=attendees))

print("\n===== TOOL 3: WEATHER API =====")
print(get_weather.run(city="Bengaluru"))

print("\n===== TOOL 4: SEATING CALCULATOR =====")
print(calculate_seating.run(attendees=attendees))

print("\n===== TOOL 5: COORDINATION CHECKLIST =====")
print(coordination_checklist.run(attendees=attendees))

print("\n===== TOOL 6: ANNOUNCEMENT GENERATOR =====")
print(create_announcement.run(
    event_name=event_name,
    date=date,
    venue=venue
))

===== TOOL 1: EVENT SCHEDULE =====
Event: AI Innovation Day
Hour 1: Welcome and registration
Hour 2: Main event activities
Hour 3: Interactive session and closing

===== TOOL 2: RESOURCE CHECKLIST =====
For 50 attendees:
Chairs: 50
Water bottles: 50
Registration badges: 50
Suggested volunteers: 3

===== TOOL 3: WEATHER API =====
Weather for Bengaluru: Temperature 27.6°C, Precipitation 0.1 mm, Wind speed 16.8 km/h

===== TOOL 4: SEATING CALCULATOR =====
Attendees: 50
Required seats including 10% extra: 55

===== TOOL 5: COORDINATION CHECKLIST =====
Coordination Checklist:
1. Confirm venue booking.
2. Arrange seating and registration.
3. Check weather conditions.
4. Test microphones and projectors.
5. Assign volunteers.
6. Keep a first-aid kit available.

===== TOOL 6: ANNOUNCEMENT GENERATOR =====
📢 COLLEGE EVENT ANNOUNCEMENT
Event: AI Innovation Day
Date: 25 October 2026
Venue: College Auditorium
All students are welcome to participate!


In [20]:
print(get_resources.run(attendees=50))

For 50 attendees:
Chairs: 50
Water bottles: 50
Registration badges: 50
Suggested volunteers: 3


In [22]:
from crewai import Agent, Task

# STEP 1: Run the six tools using Python
tool_results = {
    "schedule": create_schedule.run(
        event_name="AI Innovation Day",
        duration_hours=3
    ),
    "resources": get_resources.run(attendees=50),
    "weather": get_weather.run(city="Bengaluru"),
    "seating": calculate_seating.run(attendees=50),
    "coordination": coordination_checklist.run(attendees=50),
    "announcement": create_announcement.run(
        event_name="AI Innovation Day",
        date="25 October 2026",
        venue="College Auditorium"
    )
}

# STEP 2: Create text-only CrewAI agents
planner = Agent(
    role="Campus Event Planner",
    goal="Summarize the schedule and resources accurately",
    backstory="An experienced planner of college events.",
    llm=llm,
    tools=[],
    max_iter=2,
    verbose=True
)

coordinator = Agent(
    role="Campus Coordinator",
    goal="Summarize event arrangements using verified information",
    backstory="A coordinator responsible for smooth campus events.",
    llm=llm,
    tools=[],
    max_iter=2,
    verbose=True
)

communicator = Agent(
    role="Communication Manager",
    goal="Create a clear final event announcement",
    backstory="A communications specialist for college events.",
    llm=llm,
    tools=[],
    max_iter=2,
    verbose=True
)

# STEP 3: Task 1 - Event Planner
task1 = Task(
    description=(
        "Prepare an event planning summary for AI Innovation Day "
        "for 50 attendees, lasting 3 hours.\n"
        f"Schedule tool result:\n{tool_results['schedule']}\n"
        f"Resource tool result:\n{tool_results['resources']}\n"
        "Summarize these results accurately. Do not invent details."
    ),
    expected_output="A concise event schedule and resource summary.",
    agent=planner
)

# STEP 4: Execute Task 1
result1 = await planner.aexecute_task(task1)
print("\n===== AGENT 1: EVENT PLAN =====")
print(result1)

# STEP 5: Task 2 - Campus Coordinator

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Campus Event Planner                                                                                    │
│                                                                                                                 │
│  Task: Prepare an event planning summary for AI Innovation Day for 50 attendees, lasting 3 hours.               │
│  Schedule tool result:                                                                                          │
│  Event: AI Innovation Day                                                                                       │
│  Hour 1: Welcome and registration                                                                               │
│  Hour 2: Main event activities                                                                                  │
│  Hour 3: Interactive session and closing                                                                        │
│  Resource tool result:                                                                                          │
│  For 50 attendees:                                                                                              │
│  Chairs: 50                                                                                                     │
│  Water bottles: 50                                                                                              │
│  Registration badges: 50                                                                                        │
│  Suggested volunteers: 3                                                                                        │
│  Summarize these results accurately. Do not invent details.                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Campus Event Planner                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  AI Innovation Day, a 3-hour event for 50 attendees, is scheduled as follows:                                   │
│                                                                                                                 │
│  Hour 1: Welcome and registration                                                                               │
│  Hour 2: Main event activities                                                                                  │
│  Hour 3: Interactive session and closing                                                                        │
│                                                                                                                 │
│  Resources required for the event include:                                                                      │
│  Chairs: 50                                                                                                     │
│  Water bottles: 50                                                                                              │
│  Registration badges: 50                                                                                        │
│  Suggested volunteers: 3                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


===== AGENT 1: EVENT PLAN =====
AI Innovation Day, a 3-hour event for 50 attendees, is scheduled as follows:

Hour 1: Welcome and registration
Hour 2: Main event activities
Hour 3: Interactive session and closing

Resources required for the event include:
Chairs: 50
Water bottles: 50
Registration badges: 50
Suggested volunteers: 3


In [24]:
from crewai import Task

task2 = Task(
    description=(
        "Prepare campus arrangements for AI Innovation Day.\n\n"
        f"Event plan from Agent 1:\n{result1}\n\n"
        f"Weather report:\n{tool_results['weather']}\n\n"
        f"Seating calculation:\n{tool_results['seating']}\n\n"
        f"Coordination checklist:\n{tool_results['coordination']}\n\n"
        "Summarize the weather, seating requirements and coordination "
        "arrangements using only the supplied information."
    ),
    expected_output=(
        "A campus arrangements summary containing weather, seating "
        "and the coordination checklist."
    ),
    agent=coordinator
)

print("Task 2 recreated successfully!")

Task 2 recreated successfully!


In [25]:
result2 = await coordinator.aexecute_task(task2)

print("\n===== AGENT 2: CAMPUS ARRANGEMENTS =====")
print(result2)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Campus Coordinator                                                                                      │
│                                                                                                                 │
│  Task: Prepare campus arrangements for AI Innovation Day.                                                       │
│                                                                                                                 │
│  Event plan from Agent 1:                                                                                       │
│  AI Innovation Day, a 3-hour event for 50 attendees, is scheduled as follows:                                   │
│                                                                                                                 │
│  Hour 1: Welcome and registration                                                                               │
│  Hour 2: Main event activities                                                                                  │
│  Hour 3: Interactive session and closing                                                                        │
│                                                                                                                 │
│  Resources required for the event include:                                                                      │
│  Chairs: 50                                                                                                     │
│  Water bottles: 50                                                                                              │
│  Registration badges: 50                                                                                        │
│  Suggested volunteers: 3                                                                                        │
│                                                                                                                 │
│  Weather report:                                                                                                │
│  Weather for Bengaluru: Temperature 27.6°C, Precipitation 0.1 mm, Wind speed 16.8 km/h                          │
│                                                                                                                 │
│  Seating calculation:                                                                                           │
│  Attendees: 50                                                                                                  │
│  Required seats including 10% extra: 55                                                                         │
│                                                                                                                 │
│  Coordination checklist:                                                                                        │
│  Coordination Checklist:                                                                                        │
│  1. Confirm venue booking.                                                                                      │
│  2. Arrange seating and registration.                                                                           │
│  3. Check weather conditions.                                                                                   │
│  4. Test microphones and projectors.                                                                            │
│  5. Assign volunteers.                                                                                          │
│  6. Keep a first-aid kit available.                                                                             │
│                                                        

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Campus Coordinator                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  AI Innovation Day Campus Arrangements:                                                                         │
│                                                                                                                 │
│  Weather for Bengaluru: Temperature 27.6°C, Precipitation 0.1 mm, Wind speed 16.8 km/h                          │
│                                                                                                                 │
│  Seating Requirements:                                                                                          │
│  Attendees: 50                                                                                                  │
│  Required seats including 10% extra: 55                                                                         │
│                                                                                                                 │
│  Coordination Checklist:                                                                                        │
│  1. Confirm venue booking.                                                                                      │
│  2. Arrange seating and registration.                                                                           │
│  3. Check weather conditions.                                                                                   │
│  4. Test microphones and projectors.                                                                            │
│  5. Assign volunteers.                                                                                          │
│  6. Keep a first-aid kit available.                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


===== AGENT 2: CAMPUS ARRANGEMENTS =====
AI Innovation Day Campus Arrangements:

Weather for Bengaluru: Temperature 27.6°C, Precipitation 0.1 mm, Wind speed 16.8 km/h

Seating Requirements:
Attendees: 50
Required seats including 10% extra: 55

Coordination Checklist:
1. Confirm venue booking.
2. Arrange seating and registration.
3. Check weather conditions.
4. Test microphones and projectors.
5. Assign volunteers.
6. Keep a first-aid kit available.


In [26]:
from crewai import Task

task3 = Task(
    description=(
        "Prepare the final announcement for AI Innovation Day.\n\n"
        f"Event plan from Agent 1:\n{result1}\n\n"
        f"Campus arrangements from Agent 2:\n{result2}\n\n"
        f"Announcement template:\n{tool_results['announcement']}\n\n"
        "Create a clear, professional announcement for students.\n"
        "Event date: 25 October 2026.\n"
        "Venue: College Auditorium.\n"
        "Include the three-hour schedule, relevant arrangements, "
        "and resource information where useful.\n"
        "Use only the supplied information. Do not invent details."
    ),
    expected_output=(
        "A final college event announcement with the date, venue, "
        "schedule and relevant arrangements."
    ),
    agent=communicator
)

print("Task 3 created successfully!")

Task 3 created successfully!


In [27]:
result3 = await communicator.aexecute_task(task3)

print("\n===== AGENT 3: FINAL ANNOUNCEMENT =====")
print(result3)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Communication Manager                                                                                   │
│                                                                                                                 │
│  Task: Prepare the final announcement for AI Innovation Day.                                                    │
│                                                                                                                 │
│  Event plan from Agent 1:                                                                                       │
│  AI Innovation Day, a 3-hour event for 50 attendees, is scheduled as follows:                                   │
│                                                                                                                 │
│  Hour 1: Welcome and registration                                                                               │
│  Hour 2: Main event activities                                                                                  │
│  Hour 3: Interactive session and closing                                                                        │
│                                                                                                                 │
│  Resources required for the event include:                                                                      │
│  Chairs: 50                                                                                                     │
│  Water bottles: 50                                                                                              │
│  Registration badges: 50                                                                                        │
│  Suggested volunteers: 3                                                                                        │
│                                                                                                                 │
│  Campus arrangements from Agent 2:                                                                              │
│  AI Innovation Day Campus Arrangements:                                                                         │
│                                                                                                                 │
│  Weather for Bengaluru: Temperature 27.6°C, Precipitation 0.1 mm, Wind speed 16.8 km/h                          │
│                                                                                                                 │
│  Seating Requirements:                                                                                          │
│  Attendees: 50                                                                                                  │
│  Required seats including 10% extra: 55                                                                         │
│                                                                                                                 │
│  Coordination Checklist:                                                                                        │
│  1. Confirm venue booking.                                                                                      │
│  2. Arrange seating and registration.                                                                           │
│  3. Check weather conditions.                                                                                   │
│  4. Test microphones and projectors.                                                                            │
│  5. Assign volunteers.                                                                                          │
│  6. Keep a first-aid kit available.                    

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Communication Manager                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  📢 COLLEGE EVENT ANNOUNCEMENT                                                                                  │
│                                                                                                                 │
│  Event: AI Innovation Day                                                                                       │
│  Date: 25 October 2026                                                                                          │
│  Venue: College Auditorium                                                                                      │
│                                                                                                                 │
│  Join us for AI Innovation Day on October 25, 2026, at the College Auditorium! This 3-hour event is open to     │
│  all students and will feature a dynamic schedule, ensuring a day of learning and interaction.                  │
│                                                                                                                 │
│  **Event Schedule:**                                                                                            │
│                                                                                                                 │
│  *   **Hour 1: Welcome and Registration** - Get settled in, pick up your registration badge, and prepare for    │
│  an exciting day.                                                                                               │
│  *   **Hour 2: Main Event Activities** - Dive into the core of AI Innovation with engaging sessions.            │
│  *   **Hour 3: Interactive Session and Closing** - Participate in discussions and wrap up the day.              │
│                                                                                                                 │
│  **Arrangements and Resources:**                                                                                │
│                                                                                                                 │
│  *   **Seating:** We have arranged 55 chairs for your comfort.                                                  │
│  *   **Refreshments:** Water bottles will be provided for all attendees.                                        │
│  *   **Weather:** Bengaluru's weather on the day is expected to be around 27.6°C with minimal precipitation.    │
│                                                                                                                 │
│  Our team will ensure all arrangements, including venue setup, microphone and projector tests, and first-aid    │
│  availability, are meticulously handled for a smooth event.                                                     │
│                                                                                                                 │
│  All students are welcome to participate! We look forward to seeing you there!                                  │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────


===== AGENT 3: FINAL ANNOUNCEMENT =====
📢 COLLEGE EVENT ANNOUNCEMENT

Event: AI Innovation Day
Date: 25 October 2026
Venue: College Auditorium

Join us for AI Innovation Day on October 25, 2026, at the College Auditorium! This 3-hour event is open to all students and will feature a dynamic schedule, ensuring a day of learning and interaction.

**Event Schedule:**

*   **Hour 1: Welcome and Registration** - Get settled in, pick up your registration badge, and prepare for an exciting day.
*   **Hour 2: Main Event Activities** - Dive into the core of AI Innovation with engaging sessions.
*   **Hour 3: Interactive Session and Closing** - Participate in discussions and wrap up the day.

**Arrangements and Resources:**

*   **Seating:** We have arranged 55 chairs for your comfort.
*   **Refreshments:** Water bottles will be provided for all attendees.
*   **Weather:** Bengaluru's weather on the day is expected to be around 27.6°C with minimal precipitation.

Our team will ensure all arrange

In [28]:
final_report = f"""
SMART COLLEGE CAMPUS ASSISTANT
================================

1. EVENT PLANNER
{result1}

2. CAMPUS COORDINATOR
{result2}

3. FINAL ANNOUNCEMENT
{result3}

WORKFLOW STATUS
---------------
Three agents completed their tasks sequentially.
Six Python tools were executed.
Weather data was retrieved from the Open-Meteo API.
LLM text generation used the Nexus API.
"""

print(final_report)


SMART COLLEGE CAMPUS ASSISTANT

1. EVENT PLANNER
AI Innovation Day, a 3-hour event for 50 attendees, is scheduled as follows:

Hour 1: Welcome and registration
Hour 2: Main event activities
Hour 3: Interactive session and closing

Resources required for the event include:
Chairs: 50
Water bottles: 50
Registration badges: 50
Suggested volunteers: 3

2. CAMPUS COORDINATOR
AI Innovation Day Campus Arrangements:

Weather for Bengaluru: Temperature 27.6°C, Precipitation 0.1 mm, Wind speed 16.8 km/h

Seating Requirements:
Attendees: 50
Required seats including 10% extra: 55

Coordination Checklist:
1. Confirm venue booking.
2. Arrange seating and registration.
3. Check weather conditions.
4. Test microphones and projectors.
5. Assign volunteers.
6. Keep a first-aid kit available.

3. FINAL ANNOUNCEMENT
📢 COLLEGE EVENT ANNOUNCEMENT

Event: AI Innovation Day
Date: 25 October 2026
Venue: College Auditorium

Join us for AI Innovation Day on October 25, 2026, at the College Auditorium! This 3-ho

In [29]:
test_cases = [
    {
        "event": "AI Innovation Day",
        "duration": 3,
        "attendees": 50
    },
    {
        "event": "Coding Workshop",
        "duration": 2,
        "attendees": 30
    },
    {
        "event": "Tech Fest",
        "duration": 4,
        "attendees": 100
    }
]

for i, case in enumerate(test_cases, start=1):
    print(f"\n{'=' * 40}")
    print(f"TEST CASE {i}: {case['event']}")
    print(f"{'=' * 40}")

    print(create_schedule.run(
        event_name=case["event"],
        duration_hours=case["duration"]
    ))

    print(get_resources.run(attendees=case["attendees"]))
    print(calculate_seating.run(attendees=case["attendees"]))

print("\nTEST CASE 4: Invalid attendee count (failure case)")
print(get_resources.run(attendees=0))
print(calculate_seating.run(attendees=0))


TEST CASE 1: AI Innovation Day
Event: AI Innovation Day
Hour 1: Welcome and registration
Hour 2: Main event activities
Hour 3: Interactive session and closing
For 50 attendees:
Chairs: 50
Water bottles: 50
Registration badges: 50
Suggested volunteers: 3
Attendees: 50
Required seats including 10% extra: 55

TEST CASE 2: Coding Workshop
Event: Coding Workshop
Schedule: Welcome, main activities and closing within 2 hour(s).
For 30 attendees:
Chairs: 30
Water bottles: 30
Registration badges: 30
Suggested volunteers: 2
Attendees: 30
Required seats including 10% extra: 33

TEST CASE 3: Tech Fest
Event: Tech Fest
Hour 1: Welcome and registration
Hour 2: Main event activities
Hour 3: Interactive session and closing
For 100 attendees:
Chairs: 100
Water bottles: 100
Registration badges: 100
Suggested volunteers: 5
Attendees: 100
Required seats including 10% extra: 110

TEST CASE 4: Invalid attendee count (failure case)
Error: Attendee count must be positive.
Error: Attendee count must be positi

## Debugging Log

### Issue 1: Nexus function-calling error
- **Problem:** The tool-enabled CrewAI agent failed with a `Function calling is not enabled for this model` error.
- **Cause:** The selected `nano-banana` model did not support the required function-calling operation.
- **Solution:** Python tools were executed directly, and Nexus was used for normal text generation by text-only CrewAI agents.
- **Remaining limitation:** The original tool-enabled CrewAI crew cannot execute normally with this model configuration.

### Issue 2: Synchronous execution error
- **Problem:** Running `execute_task()` caused a runtime error in Google Colab.
- **Cause:** Synchronous execution was invoked while an event loop was already running.
- **Solution:** Used `await agent.aexecute_task(task)` for asynchronous execution.
- **Remaining limitation:** Asynchronous execution must be used consistently in this notebook.

### Issue 3: Undefined task variable
- **Problem:** Executing Agent 2 produced `NameError: name 'task2' is not defined`.
- **Cause:** The task variable was not available in the current notebook state.
- **Solution:** Re-created Task 2 and included Agent 1's output directly in its description.
- **Remaining limitation:** Notebook cells must be executed in the correct order, and required variables must exist before use.

## Conclusion

The Smart College Campus Assistant was implemented using CrewAI agents, Python tools, and the Nexus API. Three agents handled event planning, campus coordination, and student communication.

Six tools were tested, including an external Open-Meteo weather API. The workflow passed information from the Event Planner to the Campus Coordinator and then to the Communication Manager. Testing covered multiple event sizes and an invalid attendee count.

A compatibility limitation with native function calling was addressed by executing the tools directly in Python and using Nexus for normal LLM text generation. This implementation demonstrates sequential multi-agent collaboration, tool integration, external API usage, and error handling.